# NLP Workshop Notebook 01  
## From Raw Text to Basic Text Representation

**Dataset:** Abdul Majid Daryabadi English Translation of the Quran  
**Main Goal:** Understand how raw text is loaded, explored, cleaned, tokenized, and converted into a simple numerical form using **Bag of Words**.

---

## What students will learn in this notebook

By the end of this notebook, students should be able to:

1. Load a text dataset using Pandas.
2. Understand the structure of an NLP corpus.
3. Inspect basic dataset information.
4. Clean English text using Python.
5. Tokenize text into words.
6. Remove common stopwords.
7. Build a vocabulary.
8. Represent text using Bag of Words.
9. Understand the limitations of classical text representation.

---

## Workshop Flow

```text
Raw CSV File
    ↓
Load Dataset
    ↓
Explore Text Data
    ↓
Clean Text
    ↓
Tokenize Text
    ↓
Remove Stopwords
    ↓
Create Vocabulary
    ↓
Bag of Words Representation
```

# 1. Import Required Libraries

In NLP, we usually need libraries for:

- **Data loading and analysis**: `pandas`
- **Numerical computation**: `numpy`
- **Text cleaning**: `re` for regular expressions
- **Text vectorization**: `CountVectorizer` from `scikit-learn`

For this first notebook, we will avoid heavy NLP libraries so students can understand the basic process clearly.

In [1]:
import pandas as pd
import numpy as np
import re
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer

print("Libraries imported successfully.")

Libraries imported successfully.


# 2. Load the Dataset

The dataset is a CSV file containing Quranic English translation text.

For this notebook, we assume that the CSV file is available in the same folder as this notebook.

Expected file name:

```text
Abdul_Majid_Daryabadi_English_Translation.csv
```

In [2]:
CSV_PATH = "Abdul_Majid_Daryabadi_English_Translation.csv"

df = pd.read_csv(CSV_PATH)

print("Dataset loaded successfully.")
print("Shape of dataset:", df.shape)

Dataset loaded successfully.
Shape of dataset: (6236, 4)


# 3. First Look at the Dataset

Before applying any NLP technique, we must understand the dataset structure.

Important questions:

- How many rows are present?
- What are the column names?
- Which column contains the actual text?
- Are there any missing values?

In [3]:
df.head()

,Unnamed: 0,Surah,Verse,daryabadi
0,0,1,1,"In the name of Allah, the Compassionate, the M..."
1,1,1,2,"All praise unto Allah, the Lord of all the wor..."
2,2,1,3,"The Compassionate, the Merciful."
3,3,1,4,Sovereign of the Day of Requital.
4,4,1,5,Thee alone do we worship and of Thee alone do ...


In [4]:
print("Column names:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

Column names:
['Unnamed: 0', 'Surah', 'Verse', 'daryabadi']

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6236 entries, 0 to 6235
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Unnamed: 0  6236 non-null   int64 
 1   Surah       6236 non-null   int64 
 2   Verse       6236 non-null   int64 
 3   daryabadi   6236 non-null   object
dtypes: int64(3), object(1)
memory usage: 195.0+ KB


# 4. Rename and Select Useful Columns

The text column in this dataset is named `daryabadi`.

For clarity in class, we will rename it to `text`.

We will keep only important columns:

- `Surah`
- `Verse`
- `text`

In [5]:
# Remove unnecessary index column if it exists
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

# Rename translation column
if "daryabadi" in df.columns:
    df = df.rename(columns={"daryabadi": "text"})

# Keep only required columns
df = df[["Surah", "Verse", "text"]]

df.head()

,Surah,Verse,text
0,1,1,"In the name of Allah, the Compassionate, the M..."
1,1,2,"All praise unto Allah, the Lord of all the wor..."
2,1,3,"The Compassionate, the Merciful."
3,1,4,Sovereign of the Day of Requital.
4,1,5,Thee alone do we worship and of Thee alone do ...


# 5. Check Missing Values

Missing values are common in real-world datasets.

In NLP, missing text can create errors during preprocessing or vectorization.

In [6]:
df.isnull().sum()

Surah    0
Verse    0
text     0
dtype: int64

In [7]:
# Remove rows where text is missing, if any
df = df.dropna(subset=["text"]).reset_index(drop=True)

print("Dataset shape after removing missing text:", df.shape)

Dataset shape after removing missing text: (6236, 3)


# 6. Basic Corpus Statistics

In NLP:

- A **corpus** is a collection of documents.
- A **document** can be a sentence, paragraph, verse, article, or book.
- In this notebook, each Quranic verse translation is treated as one document.

Now we calculate:

- Total number of documents
- Average text length
- Minimum and maximum text length

In [8]:
df["char_length"] = df["text"].apply(len)
df["word_count_raw"] = df["text"].apply(lambda x: len(str(x).split()))

print("Total documents/verses:", len(df))
print("Average character length:", round(df["char_length"].mean(), 2))
print("Average raw word count:", round(df["word_count_raw"].mean(), 2))
print("Minimum raw word count:", df["word_count_raw"].min())
print("Maximum raw word count:", df["word_count_raw"].max())

Total documents/verses: 6236
Average character length: 134.49
Average raw word count: 24.93
Minimum raw word count: 1
Maximum raw word count: 252


In [9]:
df[["Surah", "Verse", "text", "word_count_raw"]].head(10)

,Surah,Verse,text,word_count_raw
0,1,1,"In the name of Allah, the Compassionate, the M...",9
1,1,2,"All praise unto Allah, the Lord of all the wor...",10
2,1,3,"The Compassionate, the Merciful.",4
3,1,4,Sovereign of the Day of Requital.,6
4,1,5,Thee alone do we worship and of Thee alone do ...,13
5,1,6,Guide us Thou unto the path straight,7
6,1,7,The path of those whom Thou hast favoured. Not...,21
7,2,1,Alif. Lam Mim,3
8,2,2,"This Book whereof there is no doubt, is a guid...",13
9,2,3,"Who believe in the Unseen, and establish praye...",18


# 7. Display Sample Verses

It is always important to manually inspect some text examples.

This helps us understand:

- Punctuation style
- Capitalization
- Repeated religious terms
- Sentence length
- Special characters

In [10]:
for i in range(5):
    print(f"Surah {df.loc[i, 'Surah']}, Verse {df.loc[i, 'Verse']}:")
    print(df.loc[i, "text"])
    print("-" * 80)

Surah 1, Verse 1:
In the name of Allah, the Compassionate, the Merciful.
--------------------------------------------------------------------------------
Surah 1, Verse 2:
All praise unto Allah, the Lord of all the worlds.
--------------------------------------------------------------------------------
Surah 1, Verse 3:
The Compassionate, the Merciful.
--------------------------------------------------------------------------------
Surah 1, Verse 4:
Sovereign of the Day of Requital.
--------------------------------------------------------------------------------
Surah 1, Verse 5:
Thee alone do we worship and of Thee alone do we seek help,
--------------------------------------------------------------------------------


# 8. Text Cleaning

Raw text usually contains punctuation, capitalization, extra spaces, and sometimes special symbols.

For classical NLP techniques like Bag of Words and TF-IDF, we commonly apply cleaning.

## Cleaning Steps Used Here

1. Convert text to lowercase.
2. Remove punctuation and special characters.
3. Remove extra spaces.
4. Keep only English letters and spaces.

> Note: In advanced NLP, cleaning decisions depend on the task. Sometimes punctuation and capitalization may be useful.

In [11]:
def clean_text(text):
    """
    Clean English text for classical NLP processing.
    """
    text = str(text).lower()                    # Convert to lowercase
    text = re.sub(r"[^a-z\s]", " ", text)      # Keep only letters and spaces
    text = re.sub(r"\s+", " ", text).strip()   # Remove extra spaces
    return text

# Apply cleaning
df["clean_text"] = df["text"].apply(clean_text)

df[["text", "clean_text"]].head()

,text,clean_text
0,"In the name of Allah, the Compassionate, the M...",in the name of allah the compassionate the mer...
1,"All praise unto Allah, the Lord of all the wor...",all praise unto allah the lord of all the worlds
2,"The Compassionate, the Merciful.",the compassionate the merciful
3,Sovereign of the Day of Requital.,sovereign of the day of requital
4,Thee alone do we worship and of Thee alone do ...,thee alone do we worship and of thee alone do ...


# 9. Compare Original and Cleaned Text

This step is very useful in class because students can clearly see how raw text changes after preprocessing.

In [12]:
for i in range(3):
    print("Original Text:")
    print(df.loc[i, "text"])
    print("\nCleaned Text:")
    print(df.loc[i, "clean_text"])
    print("=" * 80)

Original Text:
In the name of Allah, the Compassionate, the Merciful.

Cleaned Text:
in the name of allah the compassionate the merciful
Original Text:
All praise unto Allah, the Lord of all the worlds.

Cleaned Text:
all praise unto allah the lord of all the worlds
Original Text:
The Compassionate, the Merciful.

Cleaned Text:
the compassionate the merciful


# 10. Tokenization

Tokenization means splitting text into smaller units.

For this notebook, we perform **word tokenization** using Python's `split()` method.

Example:

```text
the lord of all the worlds
```

becomes:

```text
['the', 'lord', 'of', 'all', 'the', 'worlds']
```

In [13]:
def tokenize_text(text):
    return text.split()

df["tokens"] = df["clean_text"].apply(tokenize_text)

df[["clean_text", "tokens"]].head()

,clean_text,tokens
0,in the name of allah the compassionate the mer...,"[in, the, name, of, allah, the, compassionate,..."
1,all praise unto allah the lord of all the worlds,"[all, praise, unto, allah, the, lord, of, all,..."
2,the compassionate the merciful,"[the, compassionate, the, merciful]"
3,sovereign of the day of requital,"[sovereign, of, the, day, of, requital]"
4,thee alone do we worship and of thee alone do ...,"[thee, alone, do, we, worship, and, of, thee, ..."


# 11. Stopword Removal

Stopwords are very common words such as:

- the
- is
- and
- of
- in

These words often appear frequently but may not carry strong meaning in classical NLP tasks.

However, in religious, legal, and literary texts, stopword removal must be done carefully because small words may affect meaning.

For teaching purposes, we use a small custom stopword list.

In [14]:
custom_stopwords = {
    "the", "is", "am", "are", "was", "were", "be", "been", "being",
    "and", "or", "but", "if", "then", "than", "of", "in", "on", "at",
    "to", "for", "from", "by", "with", "as", "a", "an", "this", "that",
    "these", "those", "it", "its", "he", "she", "they", "them", "we",
    "you", "your", "our", "their", "his", "her", "do", "does", "did"
}

def remove_stopwords(tokens):
    return [word for word in tokens if word not in custom_stopwords]

df["tokens_no_stopwords"] = df["tokens"].apply(remove_stopwords)

df[["tokens", "tokens_no_stopwords"]].head()

,tokens,tokens_no_stopwords
0,"[in, the, name, of, allah, the, compassionate,...","[name, allah, compassionate, merciful]"
1,"[all, praise, unto, allah, the, lord, of, all,...","[all, praise, unto, allah, lord, all, worlds]"
2,"[the, compassionate, the, merciful]","[compassionate, merciful]"
3,"[sovereign, of, the, day, of, requital]","[sovereign, day, requital]"
4,"[thee, alone, do, we, worship, and, of, thee, ...","[thee, alone, worship, thee, alone, seek, help]"


# 12. Build Vocabulary

A vocabulary is the list of all unique words in the corpus.

In classical NLP, vocabulary size is very important because it directly affects vector size.

Large vocabulary = large sparse vectors.

In [15]:
all_tokens = []
for tokens in df["tokens_no_stopwords"]:
    all_tokens.extend(tokens)

vocab = sorted(set(all_tokens))

print("Total tokens after stopword removal:", len(all_tokens))
print("Vocabulary size:", len(vocab))
print("First 30 vocabulary words:")
print(vocab[:30])

Total tokens after stopword removal: 91869
Vocabulary size: 6368
First 30 vocabulary words:
['aad', 'aaron', 'abandon', 'abandoned', 'abase', 'abased', 'abasement', 'abasest', 'abasing', 'abate', 'abated', 'abcut', 'abhor', 'abhorence', 'abhorrence', 'abhorrent', 'abide', 'abidence', 'abider', 'abiders', 'abiding', 'abiect', 'abject', 'abjection', 'abjectness', 'able', 'abode', 'abodes', 'abolisheth', 'abominable']


# 13. Most Frequent Words

Frequency analysis is one of the oldest and most useful NLP techniques.

It helps us understand which words dominate the corpus.

In [16]:
word_freq = Counter(all_tokens)

most_common_words = word_freq.most_common(20)

pd.DataFrame(most_common_words, columns=["Word", "Frequency"])

,Word,Frequency
0,allah,2655
1,unto,2163
2,not,2112
3,ye,1851
4,who,1745
5,shall,1700
6,verily,1496
7,have,1350
8,thou,1349
9,which,1187


# 14. Bag of Words Representation

Bag of Words converts text into numerical vectors by counting word occurrences.

## Main Idea

Each document is represented by word counts.

It ignores:

- Word order
- Grammar
- Context
- Meaning relationships

But it is simple and historically very important.

In [17]:
# Use only first 10 verses for easy classroom demonstration
sample_texts = df["clean_text"].head(10).tolist()

vectorizer = CountVectorizer()
bow_matrix = vectorizer.fit_transform(sample_texts)

print("BoW matrix shape:", bow_matrix.shape)
print("Number of documents:", bow_matrix.shape[0])
print("Vocabulary size in sample:", bow_matrix.shape[1])

BoW matrix shape: (10, 64)
Number of documents: 10
Vocabulary size in sample: 64


# 15. Display Bag of Words Matrix

Rows represent documents.

Columns represent vocabulary terms.

Values represent how many times each word appears in each document.

In [18]:
bow_df = pd.DataFrame(
    bow_matrix.toarray(),
    columns=vectorizer.get_feature_names_out()
)

bow_df

,alif,all,allah,alone,and,astray,believe,book,brought,compassionate,day,do,doubt,down,establish,expend,favoured,fearing,god,guidance,guide,hast,have,help,in,indignation,is,lam,lord,merciful,mim,name,no,nor,not,of,on,out,path,praise,prayer,provided,requital,seek,sovereign,straight,that,the,thee,them,there,this,those,thou,unseen,unto,us,we,whereof,wherewith,who,whom,worlds,worship
0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,2,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,2,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0
2,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,0,0,0,0,0,0,1,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,0,0,0,2,1,0,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,2,0,0,0,0,0,0,0,0,2,0,0,0,0,0,1
5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,1,0,0,0,0,0,1,0,1,1,0,0,0,0,0,0,0
6,0,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,0,1,1,0,0,0,0,0,0,1,1,3,1,0,1,0,0,0,0,0,0,0,0,2,0,0,0,0,2,1,0,0,0,0,0,0,0,2,0,0
7,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
8,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,1,1,0,0,0,0,0,0,2,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,1,0,0,0,1,0,0,1,0,0,0,0,0
9,0,0,0,0,2,0,1,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,1,1,0,0,0,0,1,1,0,1,0,0,0,0,1,0,0,1,0,1,1,0,0,0


# 16. Bag of Words for a Single Verse

Let us inspect one verse and its vector representation.

In [19]:
verse_index = 0

print("Original verse:")
print(df.loc[verse_index, "text"])

print("\nCleaned verse:")
print(df.loc[verse_index, "clean_text"])

print("\nBag of Words vector for this verse:")
single_vector = bow_df.iloc[verse_index]
print(single_vector[single_vector > 0])

Original verse:
In the name of Allah, the Compassionate, the Merciful.

Cleaned verse:
in the name of allah the compassionate the merciful

Bag of Words vector for this verse:
allah            1
compassionate    1
in               1
merciful         1
name             1
of               1
the              3
Name: 0, dtype: int64


# 17. Limitations of Bag of Words

Bag of Words is useful but limited.

## Major Limitations

| Limitation | Explanation |
|---|---|
| No word order | "man bites dog" and "dog bites man" may look similar |
| No semantics | Similar words are treated as unrelated |
| Sparse vectors | Most values are zero |
| Large vocabulary | More words create larger vectors |
| No context | Word meaning is not understood |

## Why We Move Ahead

Because Bag of Words only counts words.  
It does not understand meaning.

This is why we later move toward:

- TF-IDF
- Word embeddings
- Sentence embeddings
- Transformers

# 18. Student Practice Tasks

Students should now try the following tasks:

## Task 1
Display verses from Surah 1 only.

## Task 2
Find the top 20 most frequent words from Surah 2.

## Task 3
Clean and tokenize any one verse manually.

## Task 4
Build Bag of Words for the first 20 verses.

## Task 5
Compare vocabulary size before and after stopword removal.

In [20]:
# Task 1 Example: Display verses from Surah 1
surah_1 = df[df["Surah"] == 1]
surah_1[["Surah", "Verse", "text"]]

,Surah,Verse,text
0,1,1,"In the name of Allah, the Compassionate, the M..."
1,1,2,"All praise unto Allah, the Lord of all the wor..."
2,1,3,"The Compassionate, the Merciful."
3,1,4,Sovereign of the Day of Requital.
4,1,5,Thee alone do we worship and of Thee alone do ...
5,1,6,Guide us Thou unto the path straight
6,1,7,The path of those whom Thou hast favoured. Not...


In [21]:
# Task 2 Example: Top 20 frequent words from Surah 2
surah_2_text = df[df["Surah"] == 2]["clean_text"]

surah_2_tokens = []
for text in surah_2_text:
    surah_2_tokens.extend(remove_stopwords(tokenize_text(text)))

pd.DataFrame(Counter(surah_2_tokens).most_common(20), columns=["Word", "Frequency"])

,Word,Frequency
0,allah,276
1,unto,225
2,ye,214
3,not,170
4,shall,151
5,who,134
6,which,115
7,verily,102
8,him,89
9,have,78


# 19. Summary of Notebook 01

In this notebook, we completed the first stage of the NLP workshop.

## We learned:

- How to load a text dataset
- How to inspect corpus structure
- How to clean text
- How to tokenize text
- How to remove stopwords
- How to build vocabulary
- How to create Bag of Words vectors
- Why Bag of Words is limited

## Next Notebook

The next notebook should cover:

# Notebook 02 — TF-IDF and Text Similarity

Topics:

- Term Frequency
- Inverse Document Frequency
- TF-IDF vectors
- Cosine similarity
- Finding similar Quranic verses
- Simple search engine using TF-IDF